In [1]:
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [3]:
print("PyTorch version:", torch.__version__)

PyTorch version: 2.15.0+cpu


# Assignment 10

The dataset records cumulative COVID-19 cases $C_t$. New daily cases arte the difference $Y_t = C_t-C_{t-1},\qquad t\geq 1$

The explanatory variables are the preceding five days of new cases assembled into a five-dimensional input vector:

$\mathbf{x}_t =\begin{bmatrix}Y_{t-1}&Y_{t-2}&Y_{t-3}&Y_{t-4}&Y_{t-5}\end{bmatrix}^{\!T}$

A supervised-learning sample is $(\mathbf{x}_t,Y_t)$ and usable samples begin at $t=5$. For $n$ usable samples, the feature matrix has dimensions $X\in\mathbb{R}^{n\times5}$ and the response vector has dimensions $\mathbf y\in\mathbb{R}^n$

In [4]:
url = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/refs/heads/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv"
covid = pd.read_csv(url)

us = covid[covid["Country/Region"] == "US"]
cases_cols = covid.columns[4:]
us_cases = us[cases_cols].sum(axis=0)

def daily_cases(cumulative):

    # calculate daily number of new cases between previous day
    daily = cumulative.diff()
    daily.iloc[0] = cumulative.iloc[0]

    data = pd.DataFrame({
        "date": daily.index,
        "Y": daily.to_numpy(dtype=float),
        "t": np.arange(len(daily))
    })

    return data.reset_index(drop=True)

def create_lags(data):
    
    df = data.copy()
    for i in range(1, 6):
        df[f"lag{i}"] = df["Y"].shift(i)
        
    df = df.dropna().reset_index(drop=True)
    return df

A = daily_cases(us_cases)
A_lag = create_lags(A)
A_lag

split = len(A_lag) // 2
X_A = A_lag[["lag1", "lag2", "lag3", "lag4", "lag5"]]
X_train = X_A.iloc[:split]
X_test = X_A.iloc[split:]
y_A = A_lag["Y"]
y_train = y_A.iloc[:split]
y_test = y_A.iloc[split:]

(289, 1147)


,Province/State,Country/Region,Lat,Long,1/22/20,1/23/20,1/24/20,1/25/20,1/26/20,1/27/20,...,2/28/23,3/1/23,3/2/23,3/3/23,3/4/23,3/5/23,3/6/23,3/7/23,3/8/23,3/9/23
0,NaN,Afghanistan,33.94,67.71,0,0,0,0,0,0,...,209322,209340,209358,209362,209369,209390,209406,209436,209451,209451
1,NaN,Albania,41.15,20.17,0,0,0,0,0,0,...,334391,334408,334408,334427,334427,334427,334427,334427,334443,334457
2,NaN,Algeria,28.03,1.66,0,0,0,0,0,0,...,271441,271448,271463,271469,271469,271477,271477,271490,271494,271496
3,NaN,Andorra,42.51,1.52,0,0,0,0,0,0,...,47866,47875,47875,47875,47875,47875,47875,47875,47890,47890
4,NaN,Angola,-11.20,17.87,0,0,0,0,0,0,...,105255,105277,105277,105277,105277,105277,105277,105277,105288,105288
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284,NaN,West Bank and Gaza,31.95,35.23,0,0,0,0,0,0,...,703228,703228,703228,703228,703228,703228,703228,703228,703228,703228
285,NaN,Winter Olympics 2022,39.90,116.41,0,0,0,0,0,0,...,535,535,535,535,535,535,535,535,535,535
286,NaN,Yemen,15.55,48.52,0,0,0,0,0,0,...,11945,11945,11945,11945,11945,11945,11945,11945,11945,11945
287,NaN,Zambia,-13.13,27.85,0,0,0,0,0,0,...,343012,343012,343079,343079,343079,343135,343135,343135,343135,343135
